In [1]:
import numpy as np
from models.seirsd import SEIRSD 
from math import ceil

np.random.seed(42)

In [2]:
class Condition:
    SUSCEPTIBLE = 0
    EXPOSED = 1
    INFECTED = 2
    RECOVERED = 3
    DEAD = 4

seirsd =  SEIRSD()

In [3]:
class Cell:
    def __init__(self):
        self.condition = Condition.SUSCEPTIBLE
        self.reset_days_count()

    def reset_days_count(self):
        self.days_exposed = 0
        self.days_infected = 0
        self.days_recovered = 0

    def get_condition(self):
        return self.condition

    def set_condition(self, condition):
        self.reset_days_count()
        self.condition = condition

    def increase_days_exposed(self):
        self.days_exposed += 1

    def increase_days_infected(self):
        self.days_infected += 1

    def increase_days_recovered(self):
        self.days_recovered += 1


    def __str__(self):
        return str(self.condition)

In [4]:
def create_population(size):
    grid = [[Cell() for j in range(size)] for i in range(size)]
    array_grid = np.array(grid, dtype=object)
    return array_grid

In [5]:
def print_grid(grid):
    size = len(grid)
    for i in range(size):
        for j in range(size):
            cell = grid[i][j]
            print(cell, end=' ')
        print()
    print()

In [6]:
def initialize_random_condition(grid, quantity, condition):
    total_susceptibles = grid.size
    flat_indices = np.random.choice(total_susceptibles, quantity, replace=False)
    indices = np.unravel_index(flat_indices, (len(grid), len(grid)))
    for row, column in zip(indices[0], indices[1]):
        grid[row, column].set_condition(condition)

In [7]:
def tick(grid, seirsd):
    beta = seirsd.get_default("beta")
    sigma  = seirsd.get_default("sigma")
    gamma = seirsd.get_default("gamma")
    alfa = seirsd.get_default("alfa")
    mu = seirsd.get_default("mu")

    days_to_infection = ceil(1/sigma)
    days_to_lose_immunity = ceil(1/alfa)
    days_to_recover = ceil(1/gamma)

    size = len(grid)
    for i in range(size):
        for j in range(size):
            cell = grid[i, j]

            match cell.get_condition():
                case Condition.SUSCEPTIBLE:
                    infected_neighbors = 0
                    for x in range(max(0, i - 1), min(size, i + 2)):
                        for y in range(max(0, j - 1), min(size, j + 2)):
                            if grid[x, y].get_condition() == Condition.INFECTED:
                                infected_neighbors += 1

                    if infected_neighbors > 0:
                        prob_infection = 1 - ((1 - beta)**infected_neighbors)
                        if (np.random.rand() < prob_infection):
                            cell.set_condition(Condition.EXPOSED)
                            
                    continue
                
                case Condition.EXPOSED: 
                    cell.increase_days_exposed()
                    if (cell.days_exposed == days_to_infection):
                        cell.set_condition(Condition.INFECTED)

                    continue
                
                case Condition.INFECTED:
                    cell.increase_days_infected()
                    prob_die = mu
                    if (np.random.rand() < prob_die):
                        cell.set_condition(Condition.DEAD)
                        continue
                    
                    if (cell.days_infected == days_to_recover):
                        cell.set_condition(Condition.RECOVERED)

                    continue
                
                case Condition.RECOVERED:
                    cell.increase_days_recovered()
                    if (cell.days_recovered == days_to_lose_immunity):
                        cell.set_condition(Condition.SUSCEPTIBLE)
                    continue

                case Condition.DEAD:
                    continue

                case _:
                    continue


In [8]:
size = 10
intial_exposeds = 5
initial_infecteds = 1
tick_count = 0

grid = create_population(size)
initialize_random_condition(grid, quantity=intial_exposeds, condition=Condition.EXPOSED)
initialize_random_condition(grid, quantity=initial_infecteds, condition=Condition.INFECTED)
print_grid(grid)

0 0 0 0 0 0 0 0 0 0 
0 0 0 0 0 0 0 0 0 0 
0 0 0 0 0 0 0 0 0 0 
0 0 0 0 0 0 0 0 0 0 
0 0 0 0 1 1 0 0 0 0 
0 0 0 1 0 0 0 0 0 0 
0 0 0 0 0 0 0 0 0 0 
1 0 0 0 0 0 0 0 0 2 
0 0 0 1 0 0 0 0 0 0 
0 0 0 0 0 0 0 0 0 0 



In [215]:

tick(grid, seirsd)
tick_count += 1
print(tick_count)
print_grid(grid)

207
2 4 2 1 2 1 4 1 1 3 
1 2 2 3 2 3 3 2 2 1 
2 2 1 1 1 1 1 1 3 2 
4 3 1 2 2 4 2 1 3 2 
2 3 1 2 3 3 1 1 4 1 
2 1 2 3 2 2 2 4 2 2 
2 1 2 2 1 1 1 2 3 2 
4 2 1 1 1 3 3 4 4 1 
1 2 1 4 1 4 4 3 2 2 
3 3 1 2 2 3 2 1 4 4 

